# Phase 4: Evaluate Extractions Against Ground Truth

CPU-only evaluation of cleaned extraction JSONL against a ground truth CSV.  
Computes per-field F1 scores, overall accuracy, and prints an Execution Summary.

**Inputs:**
- `cleaned_extractions.jsonl` — output of Stage 3 (clean)
- Ground truth CSV with an image identifier column and field columns

**Outputs:**
- `evaluation_results.jsonl` — per-image evaluation with F1, precision, recall
- Rich summary table printed in the notebook

In [ ]:
import sys
from pathlib import Path

# Ensure project root is on sys.path so `stages.*` and `common.*` resolve.
PROJECT_ROOT = Path.cwd().parent  # assumes notebook is in notebooks/
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

## Configuration

Edit the three paths below to match your run.

In [ ]:
# ---- Paths (edit these) ---- #
CLEANED_EXTRACTIONS = PROJECT_ROOT / "../evaluation_data/output/cleaned_extractions.jsonl"
GROUND_TRUTH_CSV    = PROJECT_ROOT / "../evaluation_data/bank/ground_truth_bank.csv"
OUTPUT_DIR          = PROJECT_ROOT / "../evaluation_data/output/evaluation"

# ---- Options ---- #
MATH_ENHANCEMENT = False   # Enable bank statement balance calculations
WALL_CLOCK_START = None    # Set to epoch float to report end-to-end wall clock

# Sanity check
for label, p in [("Extractions", CLEANED_EXTRACTIONS), ("Ground truth", GROUND_TRUTH_CSV)]:
    status = "OK" if p.exists() else "MISSING"
    print(f"{label}: {p.resolve()}  [{status}]")

## Run Evaluation

In [ ]:
import logging

from stages.evaluate import run

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")

results_path = run(
    cleaned_extractions_path=CLEANED_EXTRACTIONS,
    ground_truth_csv=GROUND_TRUTH_CSV,
    output_dir=OUTPUT_DIR,
    enable_math_enhancement=MATH_ENHANCEMENT,
    wall_clock_start=WALL_CLOCK_START,
)
print(f"\nResults written to: {results_path}")

## Inspect Per-Image Results

In [ ]:
import json

results = [json.loads(line) for line in results_path.read_text().splitlines() if line.strip()]

for r in results:
    f1 = r.get("median_f1", "N/A")
    acc = r.get("overall_accuracy", "N/A")
    err = r.get("error", "")
    tag = f"  ERROR: {err}" if err else ""
    print(f"{r['image_name']:40s}  median_f1={f1:<8}  accuracy={acc}{tag}")

## Field-Level Breakdown (optional)

Drill into per-field scores for a specific image.

In [ ]:
# Pick an image to inspect (change index or filter by name)
scored = [r for r in results if "field_scores" in r]
if scored:
    sample = scored[0]
    print(f"Image: {sample['image_name']}  (type: {sample['document_type']})")
    print(f"Overall — median_f1={sample['median_f1']:.3f}, accuracy={sample['overall_accuracy']:.3f}")
    print()
    for field, scores in sample["field_scores"].items():
        print(f"  {field:35s}  F1={scores['f1_score']:.3f}  P={scores['precision']:.3f}  R={scores['recall']:.3f}")
else:
    print("No scored results found — check ground truth alignment.")